# Example queries: `mapped_column` (comstock_oedi)

Auto-generated from `tests/query_snapshots/mapped_column.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [ ]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [ ]:
# Cache folder: resolve `tests/query_snapshots/comstock_oedi_cache` regardless
# of which directory Jupyter was launched from.
_CANDIDATES = [
    Path.cwd() / "tests/query_snapshots/comstock_oedi_cache",            # repo root
    Path.cwd() / "../query_snapshots/comstock_oedi_cache",               # tests/example_notebooks/
    Path.cwd() / "query_snapshots/comstock_oedi_cache",                  # tests/
]
_CACHE = next((p.resolve() for p in _CANDIDATES if p.exists()), _CANDIDATES[0].resolve())
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "comstock_amy2018_r2_2025",
    buildstock_type="comstock",
    db_schema="comstock_oedi_state_and_county",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


## `mapped_column_simple_bldg_type_group_by`

Annual electricity grouped by a MappedColumn that collapses building types into broader categories, CO only.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    restrict=[('state', ['CO'])],
    group_by=[MappedColumn(bsq=<buildstock_query.main.BuildStockQuery object at 0x152f1bef0>, name='simple_bldg_type', mapping_dict={'FullServiceRestaurant': 'Food', 'QuickServiceRestaurant': 'Food', 'RetailStandalone': 'Retail', 'RetailStripmall': 'Retail', 'PrimarySchool': 'School', 'SecondarySchool': 'School', 'SmallOffice': 'Office', 'MediumOffice': 'Office', 'LargeOffice': 'Office', 'SmallHotel': 'Lodging', 'LargeHotel': 'Lodging', 'Hospital': 'Health', 'Outpatient': 'Health', 'Warehouse': 'Warehouse'}, key=Column('in.comstock_building_type', String(), table=<bs>))],
)
result.head() if hasattr(result, 'head') else result


# shape: (7, 4)
simple_bldg_type  sample_count  units_count  electricity.total.energy_consumption..kwh
            Food         20015  6735.091005                               1.540008e+09
          Health          1420  2089.226424                               1.990316e+09
         Lodging          5608  4548.531762                               1.382815e+09
          Office         39576 22514.402048                               3.921176e+09
          Retail         44707 21431.873432                               2.990783e+09


## `mapped_column_load_factor_enduse`

MappedColumn used as an enduse rather than a group_by column, CO only. Mirrors the advanced_usage notebook pattern where a numeric mapping (per-building-type load factor) is summed across the population. Exercises the enduse path through MappedColumn.


In [ ]:
result = bsq.query(
    enduses=[MappedColumn(bsq=<buildstock_query.main.BuildStockQuery object at 0x152f1bef0>, name='bldg_load_factor', mapping_dict={'FullServiceRestaurant': 1.0, 'QuickServiceRestaurant': 0.8, 'RetailStandalone': 0.6, 'RetailStripmall': 0.5, 'PrimarySchool': 0.7, 'SecondarySchool': 0.7, 'SmallOffice': 0.4, 'MediumOffice': 0.6, 'LargeOffice': 0.9, 'SmallHotel': 0.5, 'LargeHotel': 0.8, 'Hospital': 1.0, 'Outpatient': 0.7, 'Warehouse': 0.3}, key=Column('in.comstock_building_type', String(), table=<bs>))],
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


# shape: (14, 4)
comstock_building_type  sample_count  units_count  bldg_load_factor
 FullServiceRestaurant         13461  4556.431396       4556.431396
              Hospital            73   152.003100        152.003100
            LargeHotel          2815  3317.336942       2653.869553
           LargeOffice          2271   849.927533        764.934779
          MediumOffice          8708  2418.404703       1451.042822
